# DSA 210 Projesi - Milestone 1: Detaylı EDA ve Hipotez Testi
**Hazırlayan:** Azize Keriman Kazdağlı

## 1. Giriş ve Proje Amacı
Bu analizde, Avrupa ülkelerinin ekonomik göstergelerinin (Kişi Başına GSYİH) uluslararası hava trafiği üzerindeki etkisi incelenmektedir. Veri seti 2016-2024 yıllarını kapsayan yaklaşık 650.000 uçuş kaydının temizlenmiş ve AB ülkeleri bazında ekonomik verilerle birleştirilmiş halidir.

### Araştırma Sorusu
Bir ülkenin ekonomik refah düzeyi (GDP per capita), o ülkenin hava trafiği hacmini istatistiksel olarak ne ölçüde açıklar?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats

# Görselleştirme ayarları
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 8]

# Veri setini yükleme
try:
    df = pd.read_csv('master_dataset_EU_only.csv')
    print(f"Veri seti başarıyla yüklendi. Satır sayısı: {df.shape[0]}")
    print(df.info())
except FileNotFoundError:
    print("Hata: 'master_dataset_EU_only.csv' dosyası bulunamadı.")

## 2. Keşifsel Veri Analizi (EDA)
### 2.1 Çok Değişkenli Korelasyon Analizi
Aşağıdaki ısı haritası, ekonomik değişkenler ile havacılık metrikleri arasındaki doğrusal ilişkinin gücünü göstermektedir.

In [ ]:
plt.figure(figsize=(12, 10))
corr_cols = ['FLT_TOT_1', 'GDP_pc', 'Population', 'Inbound_Per_Capita', 'Outbound_Per_Capita']
corr_matrix = df[corr_cols].corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Ekonomik Göstergeler ve Havacılık Trafiği Arasındaki İlişki')
plt.show()

### 2.2 Ekonomik Refah ve Uçuş Frekansı İlişkisi
Kişi başına düşen GSYİH (GDP per capita) arttıkça, kişilerin yurt dışına çıkış oranının (Outbound per capita) nasıl değiştiğini inceleyelim.

In [ ]:
plt.figure(figsize=(14, 7))
sns.scatterplot(data=df, x='GDP_pc', y='Outbound_Per_Capita', size='Population', hue='YEAR', palette='viridis', alpha=0.7)
sns.regplot(data=df, x='GDP_pc', y='Outbound_Per_Capita', scatter=False, color='red', label='Trend Line')

plt.title('Kişi Başına GSYİH vs. Kişi Başına Düşen Uçuş Sayısı (2016-2024)')
plt.xlabel('GDP per Capita (Current USD)')
plt.ylabel('Outbound Flights per Capita')
plt.legend(bbox_to_anchor=(1.05, 1), loc=2)
plt.show()

## 3. Hipotez Testi ve İstatistiksel Analiz
Analizimizin merkezinde yer alan "Ekonomi trafiği belirler mi?" sorusunu Pearson Korelasyon testi ile yanıtlıyoruz.

- **H0 (Null Hypothesis):** Ülke ekonomisi (GDP) ile uçuş trafiği arasında istatistiksel olarak anlamlı bir ilişki yoktur.
- **H1 (Alternative Hypothesis):** Ülke ekonomisi arttıkça uçuş trafiği de anlamlı şekilde artış gösterir.

In [ ]:
# Veri temizliği
test_df = df.dropna(subset=['GDP_pc', 'Outbound_Per_Capita'])

# İstatistiksel Test
r_val, p_val = stats.pearsonr(test_df['GDP_pc'], test_df['Outbound_Per_Capita'])

print("=== HİPOTEZ TESTİ SONUÇLARI ===")
print(f"Korelasyon Katsayısı (R): {r_val:.4f}")
print(f"P-Değeri (P-value): {p_val:.4e}")

alpha = 0.05
if p_val < alpha:
    print("\nKarar: H0 REDDEDİLDİ.")
    print(f"Yorum: P-değeri ({p_val:.4e}), 0.05 anlamlılık düzeyinden çok daha küçüktür.")
    print("Bu sonuç, ekonomi ile uçuş trafiği arasında çok güçlü ve anlamlı bir pozitif ilişki olduğunu kanıtlar.")
else:
    print("\nKarar: H0 REDDEDİLEMEDİ. Anlamlı bir ilişki bulunamadı.")

## 4. Milestone 1 Sonuçları ve Çıkarımlar
1. **Güçlü Pozitif İlişki:** GDP ve uçuş sayıları arasında beklenen pozitif ilişki doğrulanmıştır.
2. **Veri Kalitesi:** 10 yıllık verinin temizlenmesi ve AB bazında birleştirilmesi, analizin güvenilirliğini artırmıştır.
3. **Gelecek Adım:** Bir sonraki aşamada (Milestone 2), bu değişkenleri kullanarak gelecekteki trafiği tahmin eden Makine Öğrenmesi modelleri (Linear Regression, Random Forest) geliştirilecektir.